<a href="https://colab.research.google.com/github/shashikawka/268452H-CS5998-Capstone-Project/blob/main/02_data_preprocessing_and_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('/content/drive/MyDrive/CAPSTONE_PROJECT/MetroPT3.csv')

print("Shape:", df.shape)
display(df.head())

df.info()

Shape: (1516948, 17)


,Unnamed: 0,timestamp,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,COMP,DV_eletric,Towers,MPG,LPS,Pressure_switch,Oil_level,Caudal_impulses
0,0,2020-02-01 00:00:00,-0.012,9.358,9.340,-0.024,9.358,53.600,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1,10,2020-02-01 00:00:10,-0.014,9.348,9.332,-0.022,9.348,53.675,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
2,20,2020-02-01 00:00:19,-0.012,9.338,9.322,-0.022,9.338,53.600,0.0425,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
3,30,2020-02-01 00:00:29,-0.012,9.328,9.312,-0.022,9.328,53.425,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
4,40,2020-02-01 00:00:39,-0.012,9.318,9.302,-0.022,9.318,53.475,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1516948 entries, 0 to 1516947
Data columns (total 17 columns):
 #   Column           Non-Null Count    Dtype  
---  ------           --------------    -----  
 0   Unnamed: 0       1516948 non-null  int64  
 1   timestamp        1516948 non-null  object 
 2   TP2              1516948 non-null  float64
 3   TP3              1516948 non-null  float64
 4   H1               1516948 non-null  float64
 5   DV_pressure      1516948 non-null  float64
 6   Reservoirs       1516948 non-null  float64
 7   Oil_temperature  1516948 non-null  float64
 8   Motor_current    1516948 non-null  float64
 9   COMP             1516948 non-null  float64
 10  DV_eletric       1516948 non-null  float64
 11  Towers           1516948 non-null  float64
 12  MPG              1516948 non-null  float64
 13  LPS              1516948 non-null  float64
 14  Pressure_switch  1516948 non-null  float64
 15  Oil_level        1516948 non-null  float64
 16  Caudal_impulses  1

In [3]:
# Remove the unnecessary  column
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# Convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Ensure chronological order
df = df.sort_values('timestamp').reset_index(drop=True)

print("data after clean:", df.shape)
print("start:", df['timestamp'].min())
print("end:", df['timestamp'].max())

data after clean: (1516948, 16)
start: 2020-02-01 00:00:00
end: 2020-09-01 03:59:50


In [4]:
# Calculate time difference between consecutive observations
df['time_diff'] = df['timestamp'].diff()

# Mark a new segment whenever the gap is greater than 1 minute
df['new_segment'] = (
    df['time_diff'] > pd.Timedelta(minutes=1)
).astype(int)

# Assign a segment ID
df['segment_id'] = df['new_segment'].cumsum()

print("Number of continuous segments:", df['segment_id'].nunique())

Number of continuous segments: 332


##### 332 timestamp differences were greater than 1 minute.
Not deleting those rows or filling the missing periods. Instead,  mark where a new continuous segment begins.

In [9]:
#Define helper columns
#time_diff, new_segment, segment_id

segment_summary = df.groupby('segment_id').agg(
    start_time=('timestamp', 'min'),
    end_time=('timestamp', 'max'),
    observations=('timestamp', 'size')
)

print(segment_summary.head(10))
print("\nNumber of segments:", len(segment_summary))
print("Smallest segment:", segment_summary['observations'].min())
print("Largest segment:", segment_summary['observations'].max())

                    start_time            end_time  observations
segment_id                                                      
0          2020-02-01 00:00:00 2020-02-01 12:48:40          4654
1          2020-02-01 12:53:47 2020-02-01 19:40:04          2460
2          2020-02-01 23:15:33 2020-02-01 23:20:21            30
3          2020-02-02 04:34:27 2020-02-02 07:37:40          1110
4          2020-02-02 07:41:52 2020-02-05 00:32:21         23550
5          2020-02-05 00:43:00 2020-02-05 01:07:37           150
6          2020-02-05 04:28:42 2020-02-06 02:07:01          7860
7          2020-02-06 07:52:21 2020-02-06 14:08:53          2280
8          2020-02-06 14:19:23 2020-02-06 19:16:35          1800
9          2020-02-06 23:33:19 2020-02-06 23:48:02            90

Number of segments: 332
Smallest segment: 29
Largest segment: 36599


#####Temporal segmentation: Since the dataset contains occasional gaps in the timestamp sequence, gaps greater than one minute were used to separate the data into continuous segments. This resulted in 332 segments. The segmentation preserves all original observations while preventing future time-series windows from spanning large temporal gaps.

In [11]:
#chronological data split
failure_periods = [
    ('Failure 1', '2020-04-18 00:00:00', '2020-04-18 23:59:00'),
    ('Failure 2', '2020-05-29 23:30:00', '2020-05-30 06:00:00'),
    ('Failure 3', '2020-06-05 10:00:00', '2020-06-07 14:30:00'),
    ('Failure 4', '2020-07-15 14:30:00', '2020-07-15 19:00:00')
]

for name, start, end in failure_periods:
    print(name)
    print(" Start:", start)
    print(" End:  ", end)

Failure 1
 Start: 2020-04-18 00:00:00
 End:   2020-04-18 23:59:00
Failure 2
 Start: 2020-05-29 23:30:00
 End:   2020-05-30 06:00:00
Failure 3
 Start: 2020-06-05 10:00:00
 End:   2020-06-07 14:30:00
Failure 4
 Start: 2020-07-15 14:30:00
 End:   2020-07-15 19:00:00


#####should not randomly split rows. Split around the failure periods and chronological order,

In [13]:
#calculate exact positions of failures
for name, start, end in failure_periods:

    start = pd.to_datetime(start)
    end = pd.to_datetime(end)

    failure_rows = df[
        (df['timestamp'] >= start) &
        (df['timestamp'] <= end)
    ]

    print(
        name,
        "| rows:", len(failure_rows),
        "| start index:", failure_rows.index.min(),
        "| end index:", failure_rows.index.max()
    )

Failure 1 | rows: 8657 | start index: 562564 | end index: 571220
Failure 2 | rows: 2360 | start index: 840740 | end index: 843099
Failure 3 | rows: 17315 | start index: 890810 | end index: 908124
Failure 4 | rows: 1622 | start index: 1171093 | end index: 1172714


In [14]:
# Default: observation is outside a documented failure period
df['failure_period'] = 0

for name, start, end in failure_periods:
    start = pd.to_datetime(start)
    end = pd.to_datetime(end)

    mask = (
        (df['timestamp'] >= start) &
        (df['timestamp'] <= end)
    )

    df.loc[mask, 'failure_period'] = 1

print(df['failure_period'].value_counts())

print(
    "Percentage within documented failure periods:",
    df['failure_period'].mean() * 100
)

failure_period
0    1486994
1      29954
Name: count, dtype: int64
Percentage within documented failure periods: 1.974622729322297


failure_period = 0
→ Outside a documented company failure interval

failure_period = 1
→ Inside a documented company failure interval